# pass@k Run Analysis

Loads samples.csv and pass_at_k.csv from a benchmark run. Supports model and temperature sweeps.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px

RUN_NAME = None  # e.g. "tinker_temp_sweep_k1_5_10_20"

candidates = [Path("results/runs"), Path("../results/runs")]
RUN_ROOT = next((p.resolve() for p in candidates if p.exists()), candidates[0].resolve())

if RUN_NAME:
    run = RUN_ROOT / RUN_NAME
else:
    runs = sorted([p for p in RUN_ROOT.glob("*") if p.is_dir()])
    run = runs[-1] if runs else None

assert run is not None and run.exists(), f"No run found under {RUN_ROOT}"
pass_at_k_csv = run / "pass_at_k.csv"
samples_csv = run / "samples.csv"
summary_json = run / "summary.json"
assert pass_at_k_csv.exists(), f"Missing pass@k CSV: {pass_at_k_csv}"
assert samples_csv.exists(), f"Missing samples CSV: {samples_csv}"
run


In [ ]:
samples = pd.read_csv(samples_csv)
pass_at_k = pd.read_csv(pass_at_k_csv)
metric_cols = [c for c in pass_at_k.columns if c.startswith("pass@")]

aggregate = pass_at_k[pass_at_k["question_id"] == "__aggregate__"].copy()
aggregate


In [ ]:
long = aggregate.melt(
    id_vars=["model", "temperature"],
    value_vars=metric_cols,
    var_name="metric",
    value_name="pass@k",
)
long["k"] = long["metric"].str.extract(r"pass@(\d+)").astype(int)
long["temperature_label"] = "T=" + long["temperature"].astype(str)
long = long.sort_values(["model", "temperature", "k"])

fig = px.line(
    long,
    x="k",
    y="pass@k",
    color="model",
    line_dash="temperature_label",
    markers=True,
    title="Pass@k by Model and Temperature",
    labels={"k": "k", "pass@k": "pass@k", "model": "Model", "temperature_label": "Temperature"},
)
fig.update_traces(line_shape="spline")
fig.update_layout(template="plotly_white", xaxis=dict(dtick=1), yaxis=dict(range=[0, 1]))
fig.show()


In [ ]:
PLOT_K = max(int(c.split("@")[1]) for c in metric_cols)
temp_df = aggregate.sort_values(["model", "temperature"]).copy()

fig = px.line(
    temp_df,
    x="temperature",
    y=f"pass@{PLOT_K}",
    color="model",
    line_dash="model",
    markers=True,
    title=f"pass@{PLOT_K} by Temperature",
    labels={"temperature": "temperature", f"pass@{PLOT_K}": "pass@k", "model": "Model"},
)
fig.update_traces(line_shape="spline")
fig.update_layout(template="plotly_white", yaxis=dict(range=[0, 1]))
fig.show()


In [ ]:
by_question = pass_at_k[pass_at_k["question_id"] != "__aggregate__"].copy()
by_question.head()


In [ ]:
group_cols = ["model"] + (["temperature"] if "temperature" in samples.columns else [])
agg_spec = {
    "samples": ("correct", "size"),
    "correct": ("correct", "sum"),
    "errors": ("error", lambda s: s.notna().sum()),
    "avg_total_tokens": ("total_tokens", "mean"),
}
if "oracle_raw_response" in samples.columns:
    agg_spec["oracle_calls"] = ("oracle_raw_response", lambda s: s.notna().sum())

sample_stats = samples.groupby(group_cols, dropna=False).agg(**agg_spec).reset_index()
sample_stats
